## Mathmatical Trick in Attention

Below is a mathematical trick that's at the heart of efficient self-attention implementation in Transformers.

- Currently, the 8 tokens in each batch don't communicate with each other. We want them to "talk" to each other, 
- but with a specific constraint: each token should only communicate with previous tokens, not future ones. 
- For example, the token at position 5 should only access information from positions 1, 2, 3, 4, and 5 - never from positions 6, 7, or 8, since those represent future information we're trying to predict.

In [3]:
import torch
torch.manual_seed(42)

In [4]:
B, T, C = 4,8,2 #(Batch, seq_len, vocab_size)

x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

The simplest way for tokens to communicate is through averaging. 
- If I'm the 5th token, I can take my channels and average them with the channels from all previous positions (1st through 4th). This creates a feature vector that summarizes me in the context of my history.

While averaging is a weak form of interaction that loses spatial arrangement information, it's a good starting point. We'll see how to add that information back later.

In [5]:
xbow = torch.zeros(x.shape)
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev,dim=0)
x[0], xbow[0]

(tensor([[ 1.9269,  1.4873],
         [ 0.9007, -2.1055],
         [ 0.6784, -1.2345],
         [-0.0431, -1.6047],
         [-0.7521,  1.6487],
         [-0.3925, -1.4036],
         [-0.7279, -0.5594],
         [-0.7688,  0.7624]]),
 tensor([[ 1.9269,  1.4873],
         [ 1.4138, -0.3091],
         [ 1.1687, -0.6176],
         [ 0.8657, -0.8644],
         [ 0.5422, -0.3617],
         [ 0.3864, -0.5354],
         [ 0.2272, -0.5388],
         [ 0.1027, -0.3762]]))

In [6]:
torch.manual_seed(42)
a = torch.ones((3,3))
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('---')
print('b=')
print(b)
print('---')
print('c=')
print(c)

a=
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])
---
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
---
c=
tensor([[14., 16.],
        [14., 16.],
        [14., 16.]])


key trick: Instead of using a boring matrix of all ones, PyTorch provides a function called tril() wrapped in torch.ones(). This returns only the lower triangular portion of the matrix, zeroing out the upper elements.

This creates an incremental aggregation pattern where each position accumulates information from all previous positions